# Auto-label ambiguity + hallucination — 100% free (runs in Colab, no API key)
Uses an open-source model (Qwen2.5 Instruct, from Hugging Face, no login needed) running on Colab's free GPU.

**Runtime -> Change runtime type -> T4 GPU** (required; CPU is far too slow).

Input: `merged_grammar_labeled.csv` (from the grammar notebook; `merged_for_annotation.csv` also works).

| Label | AmbigQA rows | SQuAD 2.0 rows | HaluBench rows |
|---|---|---|---|
| Ambiguous + sources | dataset annotation (`multipleQAs` = ambiguous, sources = the different readings) | open LLM judge | open LLM judge |
| Hallucinated | gold answer -> `No` | gold answer -> `No` | dataset label (`FAIL` = `Yes`, `PASS` = `No`) |

The LLM also cross-checks hallucination on SQuAD/HaluBench rows. Disagreements are flagged for human review.
Small open models are noisier than big ones, so treat their output as a first pass and use the review flag.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes pandas openpyxl

## 1. Config

In [ ]:
INPUT_CSV  = "merged_grammar_labeled.csv"      # falls back to merged_for_annotation.csv
OUT_CSV    = "merged_all_labeled.csv"
OUT_XLSX   = "merged_all_labeled.xlsx"
CACHE_FILE = "llm_judgments_local.json"        # saved progress, so a disconnect doesn't lose work

# Pick a model (all free, no login):
#   "Qwen/Qwen2.5-1.5B-Instruct"  fastest, weakest
#   "Qwen/Qwen2.5-3B-Instruct"    good default on a free T4
#   "Qwen/Qwen2.5-7B-Instruct"    best quality, slower (set LOAD_4BIT = True so it fits in T4 memory)
MODEL     = "Qwen/Qwen2.5-3B-Instruct"
LOAD_4BIT = False

LIMIT        = 30       # SMOKE TEST: judge only 30 random rows first. Set to None for the full run.
JUDGE_SCOPE  = "needed" # "needed" -> skip AmbigQA rows (their labels come from the dataset)
                        # "all"    -> judge every row (cross-checks AmbigQA too, slower)
BATCH_SIZE   = 8        # lower to 4 if you get an out-of-memory error
MAX_CONTEXT_CHARS = 3000
MAX_NEW_TOKENS    = 200

TRUST_DATASET_LABELS = True   # True  -> dataset ground truth decides the final label; the LLM only cross-checks
                              # False -> the LLM decides everywhere
PREFILL_MANUAL = False        # True -> copy the final Yes/No labels into your manual columns
SEED = 42

## 2. Load data and the model

In [ ]:
import os, json, re, time
import pandas as pd
import torch

if not os.path.exists(INPUT_CSV):
    if os.path.exists("merged_for_annotation.csv"):
        INPUT_CSV = "merged_for_annotation.csv"
    else:
        from google.colab import files
        print("Upload merged_grammar_labeled.csv (or merged_for_annotation.csv) ...")
        INPUT_CSV = list(files.upload().keys())[0]

df = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)
print(INPUT_CSV, df.shape)

if not torch.cuda.is_available():
    print("WARNING: no GPU found. Runtime -> Change runtime type -> T4 GPU, then re-run. "
          "On CPU use the 1.5B model and a tiny LIMIT.")

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

kwargs = dict(device_map="auto")
if LOAD_4BIT:
    kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL, **kwargs)
else:
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, **kwargs)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, **kwargs)
model.eval()
print("Loaded", MODEL)

## 3. The open-LLM judge

In [ ]:
SYSTEM = """You are a careful data annotator. You are given a QUESTION, an optional CONTEXT passage, and an ANSWER.
Judge the ANSWER and reply with ONLY one JSON object and nothing else.

Fields:
- "is_ambiguous": "Yes" if the question has more than one reasonable interpretation, OR the answer is vague or lists competing candidates. Otherwise "No".
- "ambiguity_sources": if ambiguous, a list of 1-3 short reasons (e.g. "several entities share this name", "time period not specified"). Otherwise [].
- "is_hallucinated": if a CONTEXT is given, "Yes" when the ANSWER contradicts the context or states something the context does not support, "No" when the context supports it.
  If there is no context, "Yes" when the ANSWER is factually wrong or made up, "No" when it is correct, "Unsure" if you cannot tell.
- "hallucination_reason": one short sentence.

Example 1
QUESTION: Who wrote Hamlet?
CONTEXT: (none provided)
ANSWER: Hamlet was written by Charles Dickens.
{"is_ambiguous": "No", "ambiguity_sources": [], "is_hallucinated": "Yes", "hallucination_reason": "Hamlet was written by Shakespeare, not Dickens."}

Example 2
QUESTION: When did the war end?
CONTEXT: The treaty ended the conflict in 1648.
ANSWER: It ended in 1648.
{"is_ambiguous": "Yes", "ambiguity_sources": ["unclear which war is meant"], "is_hallucinated": "No", "hallucination_reason": "The context supports 1648."}"""

def build_prompt(row):
    ctx = row["context"].strip()[:MAX_CONTEXT_CHARS]
    ctx_part = f"CONTEXT: {ctx}" if ctx else "CONTEXT: (none provided)"
    return f"QUESTION: {row['question']}\n{ctx_part}\nANSWER: {row['answer']}"

def parse_json(text):
    m = re.search(r"\{.*?\}(?=\s*$|\s*\n)", text, re.S) or re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    # fallback: pull the fields out with regexes
    out = {}
    for key in ("is_ambiguous", "is_hallucinated"):
        mm = re.search(rf'"{key}"\s*:\s*"(\w+)"', text)
        if mm: out[key] = mm.group(1)
    mm = re.search(r'"hallucination_reason"\s*:\s*"([^"]*)"', text)
    if mm: out["hallucination_reason"] = mm.group(1)
    mm = re.search(r'"ambiguity_sources"\s*:\s*\[(.*?)\]', text, re.S)
    if mm: out["ambiguity_sources"] = re.findall(r'"([^"]+)"', mm.group(1))
    return out if out else {"error": "unparseable: " + text[:100]}

@torch.no_grad()
def generate_batch(prompts):
    chats = [tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": p}],
                                     tokenize=False, add_generation_prompt=True) for p in prompts]
    enc = tok(chats, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tok.pad_token_id)
    gen = out[:, enc["input_ids"].shape[1]:]
    return tok.batch_decode(gen, skip_special_tokens=True)

# quick test
i0 = df.index[(df["answer"].str.strip() != "") & (df["context"].str.strip() != "")][0]
raw = generate_batch([build_prompt(df.loc[i0])])[0]
print(raw, "\n->", parse_json(raw))

## 4. Run the judge (cached + resumable)

In [ ]:
cache = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}

mask = df["answer"].str.strip() != ""
if JUDGE_SCOPE == "needed" and TRUST_DATASET_LABELS:
    mask &= df["source"] != "ambigqa"
candidates = df.index[mask]
if LIMIT:
    candidates = df.loc[candidates].sample(min(LIMIT, len(candidates)), random_state=SEED).index

todo = [i for i in candidates if df.loc[i, "id"] not in cache]
prompts = {i: build_prompt(df.loc[i]) for i in todo}
todo.sort(key=lambda i: len(prompts[i]))      # similar lengths together -> less padding, faster
print(f"{len(candidates)} rows selected, {len(todo)} to judge, {len(candidates) - len(todo)} already cached")

t0 = time.time()
for s in range(0, len(todo), BATCH_SIZE):
    batch = todo[s:s + BATCH_SIZE]
    try:
        outs = generate_batch([prompts[i] for i in batch])
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        outs = []
        for i in batch:                         # retry one by one
            outs.extend(generate_batch([prompts[i]]))
    for i, o in zip(batch, outs):
        cache[df.loc[i, "id"]] = parse_json(o)
    json.dump(cache, open(CACHE_FILE, "w"))
    done = min(s + BATCH_SIZE, len(todo))
    print(f"  {done}/{len(todo)} judged  ({time.time() - t0:.0f}s)")

errors = [k for k, v in cache.items() if "error" in v]
print("Unparseable outputs:", len(errors))

## 5. Combine dataset ground truth + LLM verdicts

In [ ]:
STOP = set("the a an of in on at to and or is was were are be by for with as it its this that from".split())
def tokens(s):
    return set(re.findall(r"[a-z0-9]+", s.lower())) - STOP

def overlap(ctx, ans):
    a = tokens(ans)
    return len(a & tokens(ctx)) / len(a) if a else 0.0

def norm_yn(x, allow_unsure=False):
    x = str(x).strip().capitalize()
    if x in ("Yes", "No"):
        return x
    if allow_unsure and x == "Unsure":
        return x
    return ""

def first_str(x):
    while isinstance(x, (list, tuple)):
        x = x[0] if x else ""
    return str(x)

def ambig_sources_from_dataset(disamb_json, max_pairs=4):
    try:
        pairs = json.loads(disamb_json)
    except Exception:
        return ""
    parts = [f"reading {k+1}: \"{p['question']}\" -> {first_str(p['answer'])}" for k, p in enumerate(pairs[:max_pairs])]
    return f"{len(pairs)} different readings; " + " | ".join(parts)

keys = ["llm_is_ambiguous", "llm_ambiguity_sources", "llm_is_hallucinated", "llm_reason",
        "auto_ambiguous", "auto_ambiguity_sources", "auto_halluc_label", "auto_halluc_reason",
        "auto_label_basis", "auto_answer_context_overlap", "auto_ambig_halluc_needs_review"]
rec = {k: [] for k in keys}

for _, row in df.iterrows():
    j = cache.get(row["id"], {})
    ok = bool(j) and "error" not in j
    l_amb = norm_yn(j.get("is_ambiguous", ""))
    l_hal = norm_yn(j.get("is_hallucinated", ""), allow_unsure=True)
    srcs = j.get("ambiguity_sources", [])
    l_src = "; ".join(map(str, srcs)) if isinstance(srcs, list) else str(srcs)
    reason = j.get("hallucination_reason", "")

    src = row["source"]
    review = False
    basis = "llm" if ok else ""
    amb, amb_src, hal = l_amb, (l_src if l_amb == "Yes" else ""), l_hal

    if row["answer"].strip() == "":
        amb, amb_src, hal, basis, reason = "N/A", "", "N/A", "empty_answer", "No answer (e.g. unanswerable SQuAD question)"
    else:
        # ---- ambiguity ----
        if src == "ambigqa" and TRUST_DATASET_LABELS and row["ref_ambigqa_type"]:
            if row["ref_ambigqa_type"] == "multipleQAs":
                amb, amb_src = "Yes", ambig_sources_from_dataset(row["ref_ambigqa_disambiguated"])
            else:
                amb, amb_src = "No", ""
            basis = "dataset_label"
            if l_amb and l_amb != amb:
                review = True
        # ---- hallucination ----
        if TRUST_DATASET_LABELS:
            if src.startswith("halubench") and row["ref_original_label"] in ("PASS", "FAIL"):
                hal = "Yes" if row["ref_original_label"] == "FAIL" else "No"
                basis = "dataset_label"
                if l_hal in ("Yes", "No") and l_hal != hal:
                    review = True
            elif src in ("ambigqa", "squad_v2"):
                hal = "No"                      # gold annotator answers
                basis = "dataset_label"
                if l_hal == "Yes":
                    review = True
        if hal == "Unsure":
            review = True
        if basis == "":
            review = True                       # no judgment available (not run or unparseable)

    rec["llm_is_ambiguous"].append(l_amb);   rec["llm_ambiguity_sources"].append(l_src)
    rec["llm_is_hallucinated"].append(l_hal); rec["llm_reason"].append(reason)
    rec["auto_ambiguous"].append(amb);       rec["auto_ambiguity_sources"].append(amb_src)
    rec["auto_halluc_label"].append(hal);    rec["auto_halluc_reason"].append(reason)
    rec["auto_label_basis"].append(basis)
    rec["auto_answer_context_overlap"].append(f"{overlap(row['context'], row['answer']):.2f}" if (row["context"].strip() and row["answer"].strip()) else "")
    rec["auto_ambig_halluc_needs_review"].append("Yes" if review else "")

for k, v in rec.items():
    df[k] = v

print(df["auto_ambiguous"].value_counts(dropna=False).to_string(), "\n")
print(df["auto_halluc_label"].value_counts(dropna=False).to_string())

## 6. Sanity checks: how well does the open LLM agree with the dataset labels?

In [ ]:
judged = df[df["llm_is_hallucinated"].isin(["Yes", "No"])]

hb = judged[judged["source"].str.startswith("halubench") & judged["ref_original_label"].isin(["PASS", "FAIL"])]
if len(hb):
    print("HaluBench: dataset label (rows) vs LLM verdict (columns)")
    print(pd.crosstab(hb["ref_original_label"], hb["llm_is_hallucinated"]), "\n")

sq = judged[judged["source"] == "squad_v2"]
if len(sq):
    print("SQuAD (gold answers, should be 'No'): LLM verdicts")
    print(sq["llm_is_hallucinated"].value_counts().to_string(), "\n")

ab = df[(df["source"] == "ambigqa") & df["llm_is_ambiguous"].isin(["Yes", "No"])]
if len(ab):
    print("AmbigQA: dataset type (rows) vs LLM 'is_ambiguous' (columns)")
    print(pd.crosstab(ab["ref_ambigqa_type"], ab["llm_is_ambiguous"]), "\n")

print("Rows flagged for human review:", int((df["auto_ambig_halluc_needs_review"] == "Yes").sum()), "of", len(df))
pd.set_option("display.max_colwidth", 100)
display(df[df["auto_ambiguous"] == "Yes"][["source", "question", "auto_ambiguity_sources"]].head(8))
display(df[df["auto_halluc_label"] == "Yes"][["source", "answer", "auto_halluc_reason"]].head(8))

## 7. Save (CSV + Excel with dropdowns) and download

In [ ]:
if PREFILL_MANUAL:
    m = df["auto_ambiguous"].isin(["Yes", "No"])
    df.loc[m, "is_ambiguous"] = df.loc[m, "auto_ambiguous"]
    df.loc[m, "ambiguity_sources"] = df.loc[m, "auto_ambiguity_sources"]
    m = df["auto_halluc_label"].isin(["Yes", "No"])
    df.loc[m, "is_hallucinated"] = df.loc[m, "auto_halluc_label"]

df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

label_cols = [c for c in ["is_grammatically_correct", "is_ambiguous", "is_hallucinated"] if c in df.columns]
to_fill = label_cols + ["ambiguity_sources"]
auto_cols = [c for c in df.columns if c.startswith("auto_") or c.startswith("llm_")]
widths = {"context": 60, "question": 40, "answer": 40, "ref_ambigqa_disambiguated": 40, "ambiguity_sources": 35,
          "notes": 30, "auto_ambiguity_sources": 45, "auto_halluc_reason": 45, "llm_ambiguity_sources": 40,
          "llm_reason": 45, "auto_grammar_issues": 40}

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="annotate")
    ws = writer.sheets["annotate"]
    cols = list(df.columns)
    last = len(df) + 1

    dv = DataValidation(type="list", formula1='"Yes,No"', allow_blank=True)
    ws.add_data_validation(dv)
    for c in label_cols:
        L = get_column_letter(cols.index(c) + 1)
        dv.add(f"{L}2:{L}{last}")

    yellow = PatternFill("solid", fgColor="FFF2CC")   # you fill these
    blue   = PatternFill("solid", fgColor="DDEBF7")   # machine-generated
    for idx, name in enumerate(cols, start=1):
        cell = ws.cell(row=1, column=idx)
        cell.font = Font(bold=True)
        if name in to_fill:   cell.fill = yellow
        if name in auto_cols: cell.fill = blue
        ws.column_dimensions[get_column_letter(idx)].width = widths.get(name, 18)

    for row in ws.iter_rows(min_row=2, max_row=last):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical="top")
    ws.freeze_panes = "D2"

from google.colab import files
files.download(OUT_XLSX)
files.download(OUT_CSV)